In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from dataclasses import replace as _dc_replace

from scite.cs_design.shapes import TShape
from scite.cs_design.cross_section import CrossSection
from scite.cs_design.reinforcement import ReinforcementLayer, ReinforcementLayout
from scite.cs_design.cs_stress_strain_profile import StressStrainProfile
from scite.matmod.ec2_parabola_rectangle import EC2ParabolaRectangle
from scite.matmod.steel_reinforcement import create_steel
from scite.beam.bending.beam_deflection import BeamDeflectionAnalysis
from scite.beam.continuous import ContinuousBeamAnalysis

%matplotlib inline
plt.rcParams.update({'figure.dpi': 130, 'font.size': 10})


In [ ]:
# T-section geometry
b_f, h_f, b_w, h_w = 1150, 150, 300, 250
h   = h_w + h_f          # 400 mm total height
z_b = 40.0               # bottom bar cover to centroid [mm]
z_t = h - z_b            # top    bar cover to centroid [mm]
L   = 6000.0             # span [mm]

concrete = EC2ParabolaRectangle(f_ck=35.0, alpha_cc=0.85, gamma_c=1.5)
steel    = create_steel('B500B')

cs = CrossSection(
    shape=TShape(b_f=b_f, h_f=h_f, b_w=b_w, h_w=h_w),
    concrete=concrete,
    reinforcement=ReinforcementLayout(layers=[
        ReinforcementLayer(material=steel, A_s=804.0,  z=z_b),
        ReinforcementLayer(material=steel, A_s=1010.0, z=z_t),
    ]),
)

bda = BeamDeflectionAnalysis(
    cs=cs, L=L, load_type='dist',
    n_x=50, n_kappa=200, n_load_steps=200,
    solve_mode='both',
)
print(f'M_R (sag) = {bda.M_R:.2f} kN·m')


In [ ]:
cba = ContinuousBeamAnalysis(span_left=bda, span_right=bda, q=1.0)
s   = cba.sweep

kappa_hog  = s['kappa_hog']    # [1/mm] signed ≤ 0
M_nl_hog   = s['M_nl_hog']     # [kN·m]
q          = s['q']            # [N/mm]
M_el_hog   = s['M_el_hog']     # [kN·m]
M_nl_sag   = s['M_nl_sag_a']   # [kN·m]
M_el_sag   = s['M_el_sag_a']   # [kN·m]
kappa_sag  = s['kappa_sag_a']  # [1/mm]  κ at max-sagging-moment section
i_yield    = s['i_yield']
eps_yd     = s['eps_yd']

q_u           = q[-1]
kh_permm      = np.abs(kappa_hog) * 1e3   # |κ_hog| [‰/mm]
ks_permm      = kappa_sag * 1e3           # κ_sag   [‰/mm]
M_el_hog_qu   = abs(M_el_hog[-1])         # elastic hogging at q_u
delta_M        = cba.delta_M_u

# max sagging deflection at each sweep step
w_max = np.zeros(len(q))
for i in range(1, len(q)):
    _, w_a, _, _ = _dc_replace(cba, q=float(q[i])).w_profiles(float(M_nl_hog[i] * 1e6))
    w_max[i] = float(np.max(w_a))

print(f'Steps: {len(q)}   q_u = {q_u:.3f} N/mm   δ_M = {delta_M:.4f}')
print(f'w_max at q_u = {w_max[-1]:.2f} mm   L/250 = {L/250:.1f} mm')


In [ ]:
C_HOG, C_SAG, C_FAIL = 'black', '#005599', '#d62728'

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(q, np.abs(M_el_hog), '--', color='#666666',  lw=1.5, label='$|M_{hog}|$ elastic')
ax.plot(q, np.abs(M_nl_hog), '-',  color=C_HOG,      lw=2.0, label='$|M_{hog}|$ nonlinear')
ax.plot(q, M_el_sag,         '--', color='steelblue', lw=1.5, label='$M_{sag}$ elastic')
ax.plot(q, M_nl_sag,         '-',  color=C_SAG,       lw=2.0, label='$M_{sag}$ nonlinear')
ax.axhline(bda.M_R,           color=C_SAG,  ls=':', lw=1.2,
           label=f'$M_R$ (sag) = {bda.M_R:.1f} kN·m')
ax.axhline(abs(M_nl_hog[-1]), color=C_HOG,  ls=':', lw=1.2,
           label=f'$M_R$ (hog) = {abs(M_nl_hog[-1]):.1f} kN·m')
ax.axvline(q_u, color=C_FAIL, ls='--', lw=1.2,
           label=f'$q_{{hog,u}}$ = {q_u:.2f} N/mm')
ax.set_xlabel('$q$  [N/mm]', fontsize=11)
ax.set_ylabel('Moment  [kN·m]', fontsize=11)
ax.set_title('Redistribution: moments vs load', fontweight='bold')
ax.legend(fontsize=9); ax.grid(True, alpha=0.3, ls='--')
plt.tight_layout(); plt.show()


In [ ]:
C_YLD = '#2ca02c'
mk    = bda.mk

# x/d hogging — from M-κ neg branch
eps_bot_h = mk.eps_bot_neg_values[:len(kappa_hog)]
xd_hog    = np.array([
    StressStrainProfile(cs, kappa=float(k), eps_bottom=float(eb)).xd[2]
    for k, eb in zip(kappa_hog, eps_bot_h)
])

# x/d sagging — M_sag → κ_sag → StressStrainProfile
M_ext  = np.concatenate([[0.], mk.M_values])
k_ext  = np.concatenate([[0.], mk.kappa_values])
eb_ext = np.concatenate([[0.], mk.eps_bot_values])
xd_sag = np.full(len(kappa_hog), np.nan)
for i, M_i in enumerate(M_nl_sag):
    k_s = float(np.interp(M_i, M_ext, k_ext))
    if k_s < 1e-9:
        continue
    eb_s = float(np.interp(k_s, k_ext, eb_ext))
    xd_sag[i] = StressStrainProfile(cs, kappa=k_s, eps_bottom=eb_s).xd[2]

ku = kh_permm[-1]
kv = kh_permm[i_yield] if i_yield < len(kh_permm) else None

fig, (ax_q, ax_m, ax_xd) = plt.subplots(
    3, 1, figsize=(9, 9), sharex=True,
    gridspec_kw={'height_ratios': [2, 2, 1]}
)
plt.subplots_adjust(hspace=0, top=0.94, bottom=0.08, left=0.13, right=0.97)
ax_q.tick_params(labelbottom=False, bottom=False)
ax_m.tick_params(labelbottom=False, bottom=False)

def _vl(ax):
    if kv is not None:
        ax.axvline(kv, color=C_YLD,  ls='--', lw=1.1, zorder=0)
    ax.axvline(ku, color=C_FAIL, ls=':',  lw=1.1, zorder=0)

ax_q.set_title('κ_hog-controlled sweep', fontsize=11, fontweight='bold', pad=6)
ax_q.plot(kh_permm[:i_yield+1], q[:i_yield+1], '-', color='steelblue', lw=2, label='pre-yield')
if i_yield < len(q) - 1:
    ax_q.plot(kh_permm[i_yield:], q[i_yield:], '-', color='#9467bd', lw=2, label='post-yield')
ax_q.axhline(q_u, color=C_FAIL, ls=':', lw=1, label=f'$q_u$ = {q_u:.2f} N/mm')
_vl(ax_q); ax_q.set_ylabel('$q$  [N/mm]', fontsize=11)
ax_q.legend(fontsize=9, loc='lower right'); ax_q.grid(True, alpha=0.3, ls='--')

ax_m.plot(kh_permm, np.abs(M_nl_hog), '-', color=C_HOG, lw=2, label='$|M_{hog}|$')
ax_m.plot(kh_permm, M_nl_sag,         '-', color=C_SAG, lw=2, label='$M_{sag}$')
ax_m.axhline(bda.M_R,           color=C_SAG, ls=':', lw=1.2,
             label=f'$M_R$ sag={bda.M_R:.0f} kN·m')
ax_m.axhline(abs(M_nl_hog[-1]), color=C_HOG, ls=':', lw=1.2,
             label=f'$M_R$ hog={abs(M_nl_hog[-1]):.0f} kN·m')
ax_m.axhline(M_el_hog_qu, color='#666', ls='--', lw=1.2,
             label=f'$|M_{{el,hog}}|(q_u)$={M_el_hog_qu:.0f}  δ={delta_M:.3f}')
_vl(ax_m); ax_m.set_ylim(bottom=0, top=M_el_hog_qu * 1.12)
ax_m.set_ylabel('M  [kN·m]', fontsize=11)
ax_m.legend(fontsize=9, loc='lower right'); ax_m.grid(True, alpha=0.3, ls='--')

ax_xd.plot(kh_permm, xd_hog, '-', color=C_HOG, lw=2, label='$x/d$ hog')
ax_xd.plot(kh_permm, xd_sag, '-', color=C_SAG, lw=2, label='$x/d$ sag')
ax_xd.axhline(0.45, color='gray', ls=':', lw=1, label='EC2  $x/d=0.45$')
_vl(ax_xd)
ax_xd.set_xlabel('$|\kappa_{hog}|$  [‰/mm]', fontsize=11)
ax_xd.set_ylabel('$x/d$', fontsize=11)
ax_xd.legend(fontsize=9, loc='upper right'); ax_xd.grid(True, alpha=0.3, ls='--')
fig.align_ylabels([ax_q, ax_m, ax_xd]); plt.show()


In [ ]:
fig, ax = plt.subplots(figsize=(9, 6))

ku = kh_permm[-1]   # |κ_hog| at ultimate — x-axis cut-off

# M-κ backbone curves clipped to sweep range
k_sag_clip = mk.kappa_values[mk.kappa_values * 1e3 <= ku * 1.05] * 1e3
M_sag_clip = mk.M_values[:len(k_sag_clip)]
k_hog_clip = np.abs(mk.kappa_neg_values)[np.abs(mk.kappa_neg_values) * 1e3 <= ku * 1.05] * 1e3
M_hog_clip = np.abs(mk.M_neg_values[:len(k_hog_clip)])
ax.plot(k_sag_clip, M_sag_clip, '-', color='#cccccc', lw=1.5, label=r'M-$\kappa$  sagging branch')
ax.plot(k_hog_clip, M_hog_clip, '-', color='#aaaaaa', lw=1.5, label=r'M-$\kappa$  hogging branch')

# Sweep paths — each section against its own curvature
ax.plot(kh_permm, np.abs(M_nl_hog), '-', color=C_HOG, lw=2.5,
        label=r'$|M_{hog}|$ vs $|\kappa_{hog}|$')
ax.plot(ks_permm, M_nl_sag,         '-', color=C_SAG, lw=2.5,
        label=r'$M_{sag}$  vs $\kappa_{sag}$')

# Capacity reference lines
ax.axhline(bda.M_R,           color=C_SAG, ls=':', lw=1.2,
           label=f'$M_R$ sag = {bda.M_R:.1f} kN·m')
ax.axhline(abs(M_nl_hog[-1]), color=C_HOG, ls=':', lw=1.2,
           label=f'$M_R$ hog = {abs(M_nl_hog[-1]):.1f} kN·m')

# Final-state markers
ax.scatter([kh_permm[-1]], [abs(M_nl_hog[-1])],
           color=C_FAIL, zorder=5, s=70,
           label=f'$q_{{hog,u}}$: hog {abs(M_nl_hog[-1]):.1f} kN·m  '
                 f'sag {M_nl_sag[-1]:.1f} kN·m')
ax.scatter([ks_permm[-1]], [M_nl_sag[-1]], color=C_FAIL, zorder=5, s=70)

ax.set_xlabel('κ  [‰/mm]', fontsize=11)
ax.set_ylabel('M  [kN·m]', fontsize=11)
ax.set_title('M-κ paths: hogging (control) and sagging (compatible)',
             fontweight='bold')
ax.legend(fontsize=9, loc='lower right')
ax.grid(True, alpha=0.3, ls='--')
ax.set_xlim(left=0, right=ku * 1.05)
ax.set_ylim(bottom=0)
plt.tight_layout(); plt.show()


In [ ]:
w_lim = L / 250   # serviceability limit [mm]

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(w_max, q, '-', color=C_SAG, lw=2, label='$q(w_{\\max})$  nonlinear')
ax.axvline(w_lim, color=C_FAIL, ls='--', lw=1.5,
           label=f'$L/250$ = {w_lim:.0f} mm')
# mark q at L/250
q_at_lim = float(np.interp(w_lim, w_max, q))
ax.scatter([w_lim], [q_at_lim], color=C_FAIL, zorder=5, s=60,
           label=f'$q(L/250)$ = {q_at_lim:.2f} N/mm')
ax.set_xlabel('Max sagging deflection  $w$  [mm]', fontsize=11)
ax.set_ylabel('$q$  [N/mm]', fontsize=11)
ax.set_title('Load–deflection response  (max sagging span)', fontweight='bold')
ax.legend(fontsize=9); ax.grid(True, alpha=0.3, ls='--')
ax.set_xlim(left=0); ax.set_ylim(bottom=0)
plt.tight_layout(); plt.show()
